In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import time
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from src.utils_jerem import evaluate, log_result
from src.data_jerem import load_raw, build


X_tr, Y_tr, X_te, Y_te, A = load_raw()
print(X_tr.shape, Y_tr.shape, X_te.shape, Y_te.shape, A.shape)

(1261, 36, 48) (1261, 36) (840, 36, 48) (840, 36) (36, 36)


In [7]:
for k in (0, 1, 2):
    corr = [np.corrcoef(X_tr[k:, :, j].ravel(), Y_tr[:len(Y_tr) - k].ravel())[0, 1] for j in range(48)]
    corr = np.nan_to_num(corr)
    j = int(np.argmax(corr)); print(f"décalage {k} : colonne {j}, corrélation {corr[j]:.5f}")

décalage 0 : colonne 9, corrélation 0.96227
décalage 1 : colonne 9, corrélation 0.98103
décalage 2 : colonne 9, corrélation 1.00000


In [9]:
print(np.allclose(X_tr[2:, :, 9], Y_tr[:-2]))   # doit afficher True
for j in range(10):
    for k in range(1, 20):
        if np.allclose(X_tr[k:, :, j], Y_tr[:-k]):
            print(f"colonne {j} = volume {k} quarts d'heure avant la cible")

d = build("base")
print(evaluate(d["y_val"], d["X_val"][:, 9]))

True
colonne 0 = volume 11 quarts d'heure avant la cible
colonne 1 = volume 10 quarts d'heure avant la cible
colonne 2 = volume 9 quarts d'heure avant la cible
colonne 3 = volume 8 quarts d'heure avant la cible
colonne 4 = volume 7 quarts d'heure avant la cible
colonne 5 = volume 6 quarts d'heure avant la cible
colonne 6 = volume 5 quarts d'heure avant la cible
colonne 7 = volume 4 quarts d'heure avant la cible
colonne 8 = volume 3 quarts d'heure avant la cible
colonne 9 = volume 2 quarts d'heure avant la cible
{'RMSE': 0.05589856254155216, 'MAE': 0.03661767441898628}


In [4]:
Xf = X_tr.reshape(-1, 48)       # on aplatit instants × capteurs
for j in range(10, 48):
    u = np.unique(Xf[:, j])
    print(j, len(u), u[:4])

10 2 [0. 1.]
11 2 [0. 1.]
12 2 [0. 1.]
13 2 [0. 1.]
14 2 [0. 1.]
15 2 [0. 1.]
16 2 [0. 1.]
17 2 [0. 1.]
18 2 [0. 1.]
19 2 [0. 1.]
20 2 [0. 1.]
21 2 [0. 1.]
22 2 [0. 1.]
23 2 [0. 1.]
24 2 [0. 1.]
25 2 [0. 1.]
26 2 [0. 1.]
27 2 [0. 1.]
28 2 [0. 1.]
29 2 [0. 1.]
30 2 [0. 1.]
31 2 [0. 1.]
32 2 [0. 1.]
33 2 [0. 1.]
34 2 [0. 1.]
35 2 [0. 1.]
36 2 [0. 1.]
37 2 [0. 1.]
38 2 [0. 1.]
39 5 [1. 2. 3. 4.]
40 2 [0. 1.]
41 2 [0. 1.]
42 2 [0. 1.]
43 2 [0. 1.]
44 2 [0. 1.]
45 2 [0. 1.]
46 2 [0. 1.]
47 2 [0. 1.]


In [5]:
for a, b, nom in [(10, 17, "jour ?"), (17, 41, "heure ?"), (41, 45, "direction ?")]:
    print(nom, np.unique(Xf[:, a:b].sum(1)))     # attendu : [1.]

jour ? [1. 2.]
heure ? [1. 2. 3. 4. 5. 6. 7.]
direction ? [0. 1. 2.]


In [6]:
Xf = X_tr.reshape(-1, 48)
bin_cols = [j for j in range(10, 48) if set(np.unique(Xf[:, j])) <= {0., 1.}]

blocks = []
for a in range(10, 48):
    for b in range(a + 2, 49):
        if all(c in bin_cols for c in range(a, b)) and (Xf[:, a:b].sum(1) == 1).all():
            blocks.append((a, b))
print(blocks)

static = [j for j in range(10, 48)
          if all(len(np.unique(X_tr[:, s, j])) == 1 for s in range(36))]
print("colonnes fixes par capteur :", static)
print("colonnes qui varient dans le temps :", [j for j in range(10, 48) if j not in static])

print(np.unique(X_tr[0, :, 39], return_counts=True))

[(10, 15), (15, 39), (40, 44), (43, 46), (44, 48)]
colonnes fixes par capteur : [39, 40, 41, 42, 43, 44, 45, 46, 47]
colonnes qui varient dans le temps : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
(array([1., 2., 3., 4., 5.]), array([ 1,  3, 10, 20,  2]))


In [12]:
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

cols = ([f"lag_{k}" for k in range(11, 1, -1)]      # colonnes 0 à 9
        + [f"jour_{i}" for i in range(5)]           # 10 à 14
        + [f"heure_{h}" for h in range(24)]         # 15 à 38
        + ["voies"]                                 # 39
        + [f"bloc40_{i}" for i in range(4)]         # 40 à 43
        + [f"bloc44_{i}" for i in range(4)])        # 44 à 47
assert len(cols) == 48

Xf = X_tr.reshape(-1, 48)
df = pd.DataFrame(Xf, columns=cols)
df["instant"] = np.repeat(np.arange(len(X_tr)), 36)
df["capteur"] = np.tile(np.arange(36), len(X_tr))
df["jour"] = Xf[:, 10:15].argmax(1)                 # 0 à 4 (version lisible)
df["heure"] = Xf[:, 15:39].argmax(1)                # 0 à 23
df["y"] = Y_tr.reshape(-1)

print(df.shape)                                     # attendu : (45396, 56)

(45396, 53)


In [13]:
df[df.capteur == 0][["instant", "jour", "heure", "lag_3", "lag_2", "y"]].head(15)

,instant,jour,heure,lag_3,lag_2,y
0,0,0,0,0.054647,0.050911,0.050911
36,1,0,0,0.050911,0.042971,0.047641
72,2,0,0,0.042971,0.050911,0.044839
108,3,0,0,0.050911,0.047641,0.059785
144,4,0,1,0.047641,0.044839,0.062588
180,5,0,1,0.044839,0.059785,0.063989
216,6,0,1,0.059785,0.062588,0.088277
252,7,0,1,0.062588,0.063989,0.113498
288,8,0,2,0.063989,0.088277,0.122840
324,9,0,2,0.088277,0.113498,0.125175


In [14]:
print(len(np.unique(X_tr[0, :, 39:48], axis=0)))

17


In [15]:
static = X_tr[0, :, 39:48]
_, inv = np.unique(static, axis=0, return_inverse=True)
inv = inv.ravel()

groups = {}
for s, g in enumerate(inv):
    groups.setdefault(int(g), []).append(s)

niveau = Y_tr.mean(0)          # volume moyen de chaque capteur
for g, sensors in groups.items():
    if len(sensors) > 1:
        print(sensors, niveau[sensors].round(3))

[0, 2, 3] [0.248 0.215 0.248]
[5, 6] [0.315 0.357]
[8, 9, 10, 12, 13, 14] [0.37  0.44  0.386 0.404 0.388 0.504]
[15, 16] [0.201 0.273]
[17, 18, 19, 20, 21] [0.294 0.306 0.399 0.376 0.419]
[22, 23] [0.183 0.167]
[28, 29, 35] [0.125 0.349 0.35 ]
[30, 31, 34] [0.356 0.341 0.35 ]
[32, 33] [0.11 0.24]


In [ ]:
for name, X in [("train", X_tr), ("test", X_te)]:
    day = X[:, 0, 10:15].argmax(1)        # 0 à 4
    hour = X[:, 0, 15:39].argmax(1)       # 0 à 23
    chg = np.flatnonzero(np.diff(day) != 0) + 1
    print(name, "| 1er pas : jour", day[0], "heure", hour[0])
    print("  écarts entre changements de jour :", np.unique(np.diff(chg)))
    print("  séquence des jours :", day[np.r_[0, chg]])
    print("  sauts d'heure :", np.unique(np.diff(hour) % 24))

train | 1er pas : jour 0 heure 0
  écarts entre changements de jour : [96]
  séquence des jours : [0 1 2 3 4 0 1 2 3 4 0 1 2 3]
  sauts d'heure : [0 1]
test | 1er pas : jour 3 heure 3
  écarts entre changements de jour : [94 96]
  séquence des jours : [3 4 0 1 2 3 4 0 1]
  sauts d'heure : [0 1]


In [17]:
print(np.allclose(X_te[0, :, :9], X_tr[-1, :, 1:10]))     # True : le test suit le train
print(np.allclose(X_tr[0, :, :9], X_te[-1, :, 1:10]))     # True : le train suivrait le test

True
False


In [18]:
day  = X_te[:, 0, 10:15].argmax(1)
hour = X_te[:, 0, 15:39].argmax(1)
bounds = np.r_[0, np.flatnonzero(np.diff(day) != 0) + 1, len(day)]
print("longueur de chaque jour :", np.diff(bounds))
i = int(np.argmax(np.diff(bounds) == 94))
print("quarts d'heure par heure (jour de 94 pas) :", np.bincount(hour[bounds[i]:bounds[i + 1]], minlength=24))
ok = np.isclose(X_te[2:, :, 9], Y_te[:-2]).all(1)
print("lags et cible cohérents sur tout le test :", ok.all(), np.flatnonzero(~ok))

longueur de chaque jour : [83 96 96 96 96 94 96 96 87]
quarts d'heure par heure (jour de 94 pas) : [4 4 4 4 4 4 4 4 4 4 4 4 3 3 4 4 4 4 4 4 4 4 4 4]
lags et cible cohérents sur tout le test : True []


In [19]:
print("Y train : min %.3f max %.3f moyenne %.3f écart-type %.3f" % (Y_tr.min(), Y_tr.max(), Y_tr.mean(), Y_tr.std()))
print("Y test  : min %.3f max %.3f moyenne %.3f écart-type %.3f" % (Y_te.min(), Y_te.max(), Y_te.mean(), Y_te.std()))
print("part de zéros :", (Y_tr == 0).mean())

Y train : min 0.000 max 1.000 moyenne 0.304 écart-type 0.200
Y test  : min 0.000 max 0.998 moyenne 0.321 écart-type 0.204
part de zéros : 8.811349017534584e-05


In [20]:
d = build("base")                  # scaled=False : la colonne 9 doit être en valeurs réelles
y    = d["y_val"]                  # vraies valeurs
pred = d["X_val"][:, 9]            # prédiction : le dernier volume connu
err  = y - pred

print(len(y))                          # 9108 lignes (253 instants × 36 capteurs)
print(np.sqrt(np.mean(err ** 2)))      # 0.0559 → RMSE
print(np.mean(np.abs(err)))            # 0.0366 → MAE

9108
0.05589856254155216
0.03661767441898628


In [2]:
d = build("base")                       # non normalisé : la colonne 9 reste en valeurs réelles
print("formes :", d["X_train"].shape, d["X_val"].shape, d["X_test"].shape)
print("écart-type de y_val :", round(d["y_val"].std(), 4))

# Baselines
m = evaluate(d["y_val"], np.full_like(d["y_val"], d["y_train"].mean()))
print("moyenne     :", m); log_result("moyenne", "base", "mean", "val", m)
m = evaluate(d["y_val"], d["X_val"][:, 9])
print("persistance :", m); log_result("persistance", "base", "col9", "val", m)

# Modèles de fumée
for name, mdl in [("ridge", Ridge()),
                  ("rf", RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=0)),
                  ("xgb", XGBRegressor(n_estimators=100, random_state=0))]:
    t0 = time.time()
    mdl.fit(d["X_train"], d["y_train"])
    m = evaluate(d["y_val"], mdl.predict(d["X_val"]))
    print(f"{name:12s}:", m, f"{time.time() - t0:.1f}s")
    log_result(name, "base", "défaut", "val", m)

formes : (36288, 48) (9108, 48) (30240, 48)
écart-type de y_val : 0.1995
moyenne     : {'RMSE': 0.20001936504796916, 'MAE': 0.16790347873286152}
persistance : {'RMSE': 0.05589856254155216, 'MAE': 0.03661767441898628}
ridge       : {'RMSE': 0.04658897123098305, 'MAE': 0.02976476690562463} 0.0s
rf          : {'RMSE': 0.03861786780314621, 'MAE': 0.023552878458651866} 2.1s
xgb         : {'RMSE': 0.038127465253998495, 'MAE': 0.023424417046662042} 0.3s
